# Installing Ollama

In [5]:
!nvidia-smi

Thu Sep  3 13:04:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!curl -fsSL https://ollama.com/install.sh | sh

!apt-get update -qq
!apt-get install -y zstd

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                           13.8%#####                                               38.9%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 220 not upgr

## Running Ollama server

In [7]:
%%bash

# Stop any old Ollama processes
pkill -f "ollama serve" 2>/dev/null || true
pkill -f "ollama_supervisor" 2>/dev/null || true

# ============================================================
# Ollama configuration
# ============================================================

# Make BOTH Tesla T4 GPUs visible to Ollama
export CUDA_VISIBLE_DEVICES=0,1

# Keep models loaded indefinitely
export OLLAMA_KEEP_ALIVE=-1

# Ollama server
export OLLAMA_HOST=127.0.0.1:11434
export OLLAMA_ORIGINS="*"

# ============================================================
# Create supervisor
# ============================================================

cat > /tmp/ollama_supervisor.sh <<'EOF'
#!/bin/bash

# IMPORTANT: expose both Kaggle GPUs to Ollama
export CUDA_VISIBLE_DEVICES=0,1

export OLLAMA_KEEP_ALIVE=-1
export OLLAMA_HOST=127.0.0.1:11434
export OLLAMA_ORIGINS="*"

while true; do
    echo "$(date): Starting Ollama..." >> /tmp/ollama_supervisor.log

    ollama serve >> /tmp/ollama.log 2>&1

    EXIT_CODE=$?

    echo "$(date): Ollama exited with code $EXIT_CODE. Restarting..." \
        >> /tmp/ollama_supervisor.log

    sleep 2
done
EOF

chmod +x /tmp/ollama_supervisor.sh

# ============================================================
# Start supervisor
# ============================================================

nohup /tmp/ollama_supervisor.sh \
    > /tmp/ollama_supervisor.log 2>&1 < /dev/null &

# ============================================================
# Wait for Ollama
# ============================================================

echo "Waiting for Ollama..."

for i in {1..30}; do
    if curl -sf http://127.0.0.1:11434/api/version > /tmp/ollama_version.json; then
        break
    fi
    sleep 1
done

# ============================================================
# Verify Ollama
# ============================================================

echo
echo "=== Ollama ==="

cat /tmp/ollama_version.json 2>/dev/null || {
    echo "Ollama failed to start"
    echo
    echo "=== Ollama log ==="
    tail -50 /tmp/ollama.log
    exit 1
}

echo
echo "=== Process ==="
ps aux | grep '[o]llama'

echo
echo "=== Port ==="
ss -ltnp | grep 11434 || true

echo
echo "=== GPUs visible to Ollama ==="
echo "CUDA_VISIBLE_DEVICES=$CUDA_VISIBLE_DEVICES"
nvidia-smi --query-gpu=index,name,memory.total --format=csv

# ============================================================
# Pull model
# ============================================================

echo
echo "=== Pulling Qwen3 8B ==="
ollama pull qwen3:8b

# ============================================================
# Show models
# ============================================================

echo
echo "=== Models ==="
ollama list

Waiting for Ollama...

=== Ollama ===
{"version":"0.33.2"}
=== Process ===
root        1932  0.0  0.0   7376  3536 ?        S    13:05   0:00 /bin/bash /tmp/ollama_supervisor.sh
root        1935  1.6  0.1 2255752 38840 ?       Sl   13:05   0:00 ollama serve

=== Port ===
LISTEN 0      4096       127.0.0.1:11434      0.0.0.0:*    users:(("ollama",pid=1935,fd=4))      

=== GPUs visible to Ollama ===
CUDA_VISIBLE_DEVICES=0,1
index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB

=== Pulling Qwen3 8B ===

=== Models ===
NAME        ID              SIZE      MODIFIED               
qwen3:8b    500a1f067a9f    5.2 GB    Less than a second ago    


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling a3de86cd1c13:   0% ▕                  ▏  21 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   2% ▕                  ▏ 102 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   3% ▕                  ▏ 144 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   5% ▕                  ▏ 236 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   6% ▕█                 ▏ 323 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   8% ▕█                 ▏ 412 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   9% ▕█                 ▏ 461 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:  11% ▕█                 ▏ 551 MB/5.2 GB                  pulling manifes

> Checking the server

In [8]:
import requests
import pprint

r = requests.get("http://127.0.0.1:11434/api/tags")
t = requests.get("http://127.0.0.1:11434/api/version")

print(r.status_code)
pprint.pp(r.json())
print(t.status_code)
pprint.pp(t.json())

200
{'models': [{'name': 'qwen3:8b',
             'model': 'qwen3:8b',
             'modified_at': '2026-09-03T13:06:08.855677814Z',
             'size': 5225388164,
             'digest': '500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41',
             'details': {'parent_model': '',
                         'format': 'gguf',
                         'family': 'qwen3',
                         'families': ['qwen3'],
                         'parameter_size': '8.2B',
                         'quantization_level': 'Q4_K_M',
                         'context_length': 40960,
                         'embedding_length': 4096},
             'capabilities': ['completion', 'tools', 'thinking']}]}
200
{'version': '0.33.2'}


> Checking Structured Output using ollama

In [9]:
!pip install -q ollama pydantic

In [10]:
from ollama import Client
from pydantic import BaseModel, Field
from pprint import pp

class BISResponse(BaseModel):
    answer: str
    confidence: float = Field(
        ge=0.0,
        le=1.0
    )


client = Client(
    host="http://127.0.0.1:11434"
)


response = client.chat(
    model="qwen3:8b",

    messages=[
        {
            "role": "user",
            "content": "What is BIS?"
        }
    ],

    format=BISResponse.model_json_schema(),

    options={
        "temperature": 0,
    }
)


pp(response["message"]["content"])

('{"answer": "BIS can refer to different organizations depending on the '
 'context. The most prominent one is the **Bank for International Settlements '
 '(BIS)**, which is an international financial institution headquartered in '
 'Basel, Switzerland. It serves as a forum for central banks and monetary '
 'authorities to collaborate on global financial stability, monetary policy, '
 'and regulatory standards. Key functions include:\\n\\n- Facilitating '
 'cooperation among central banks.\\n- Conducting research on global financial '
 'systems.\\n- Setting standards for financial regulations (e.g., Basel '
 'Accords).\\n- Acting as a bank for central banks (e.g., holding reserves for '
 'member institutions).\\n\\nOther possible meanings include:\\n- **Bureau of '
 'Indian Standards (BIS)**: A standards organization in India that sets '
 'quality and safety standards for products.\\n- **British Indian Society '
 "(BIS)**: A historical cultural organization in the UK.\\n\\nIf you're "


In [11]:
result = BISResponse.model_validate_json(
    response["message"]["content"]
)

print(result)
print(result.answer)
print(result.confidence)

answer="BIS can refer to different organizations depending on the context. The most prominent one is the **Bank for International Settlements (BIS)**, which is an international financial institution headquartered in Basel, Switzerland. It serves as a forum for central banks and monetary authorities to collaborate on global financial stability, monetary policy, and regulatory standards. Key functions include:\n\n- Facilitating cooperation among central banks.\n- Conducting research on global financial systems.\n- Setting standards for financial regulations (e.g., Basel Accords).\n- Acting as a bank for central banks (e.g., holding reserves for member institutions).\n\nOther possible meanings include:\n- **Bureau of Indian Standards (BIS)**: A standards organization in India that sets quality and safety standards for products.\n- **British Indian Society (BIS)**: A historical cultural organization in the UK.\n\nIf you're referring to a specific context, please clarify!" confidence=0.95
B

# Installing Cloudflare

In [12]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

!cloudflared --version

Selecting previously unselected package cloudflared.
(Reading database ... 125207 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.8.3) ...
Setting up cloudflared (2026.8.3) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared version 2026.8.3 (built 2026-08-31-10:04 UTC)


In [13]:
# Checking the server is alive or not

!curl http://127.0.0.1:11434/api/version

{"version":"0.33.2"}

## Tunneling the ollama server through cloudflare

In [14]:
import subprocess
import time
import re
import os

tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:11434",
        "--http-host-header",
        "localhost:11434",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = None

for _ in range(30):
    line = tunnel_process.stdout.readline()

    if line:
        print(line, end="")

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:
            tunnel_url = match.group(0)
            break

    time.sleep(0.2)

print("Tunnel:", tunnel_url)

os.environ["TUNNEL_URL"] = tunnel_url

2026-09-03T13:08:02Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-03T13:08:02Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-03T13:08:08Z INF +--------------------------------------------------------------------------------------------+
2026-09-03T13:08:08Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-03T13:08:08Z INF |  https://roulette-cover-suite-disciplinary.trycloudfla

In [15]:
for _ in range(20):
    line = tunnel_process.stdout.readline()

    if line:
        print(line, end="")

2026-09-03T13:08:08Z INF +--------------------------------------------------------------------------------------------+
2026-09-03T13:08:08Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
2026-09-03T13:08:08Z INF Version 2026.8.3 (Checksum f29324fe934d1e100617484c78deef803c4dc2cd351d645bbde42e96b4fccc5e)
2026-09-03T13:08:08Z INF GOOS: linux, GOVersion: go1.26.4, GoArch: amd64
2026-09-03T13:08:08Z INF Settings: map[ha-connections:1 http-host-header:localhost:11434 protocol:quic url:http://127.0.0.1:11434]
2026-09-03T13:08:08Z INF cloudflared will not automatically update if installed by a package manager.
2026-09-03T13:08:08Z INF Generated Connector ID: c0dbdb78-80da-4266-86ef-da23c9df6791
2026-09-03T13:08:08Z INF Initial protocol quic
2026-09-03T13:08:08Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
2026-09-03T13:08:08Z INF ICMP proxy will use

In [16]:
!echo "=== Runtime ==="
!uptime

!echo "=== Ollama process ==="
!ps aux | grep '[o]llama'

!echo "=== Port 11434 ==="
!ss -ltnp | grep 11434 || true

!echo "=== Last Ollama logs ==="
!tail -50 /tmp/ollama.log

=== Runtime ===
 13:08:14 up 8 min,  0 users,  load average: 1.08, 1.00, 0.49
=== Ollama process ===
root        1932  0.0  0.0   7376  3536 ?        S    13:05   0:00 /bin/bash /tmp/ollama_supervisor.sh
root        1935 22.3  0.1 3067636 54720 ?       Sl   13:05   0:40 ollama serve
root        2152 95.7  3.6 50690824 1200572 ?    Sl   13:06   1:53 /usr/local/lib/ollama/llama-server --model /root/.ollama/models/blobs/sha256-a3de86cd1c132c822487ededd47a324c50491393e6565cd14bafa40d0b8e686f --port 40751 --host 127.0.0.1 --no-webui --offline -c 32768 -np 1 --log-verbosity 4 --no-log-prefix --no-log-timestamps --no-jinja --chat-template chatml --flash-attn auto -b 1024 -ub 1024 --split-mode none --main-gpu 0 --context-shift --keep 4
=== Port 11434 ===
LISTEN 0      4096       127.0.0.1:11434      0.0.0.0:*    users:(("ollama",pid=1935,fd=4))       
=== Last Ollama logs ===
srv  get_availabl: updating prompt cache
srv          load:  - looking for better prompt, base f_keep = -1.000, f_sim =

In [17]:
!tail -30 /tmp/ollama_supervisor.log

Thu Sep  3 01:05:11 PM UTC 2026: Starting Ollama...


In [18]:
!curl -i $TUNNEL_URL/api/version

HTTP/2 200 
date: Thu, 03 Sep 2026 13:08:16 GMT
content-type: application/json; charset=utf-8
content-length: 20
cf-ray: a354fb6eda52fffb-AMS
cf-cache-status: DYNAMIC
server: cloudflare

{"version":"0.33.2"}

In [19]:
!curl -i $TUNNEL_URL/api/version

!curl -i \
  -H "Origin: $TUNNEL_URL" \
  $TUNNEL_URL/api/version

HTTP/2 200 
date: Thu, 03 Sep 2026 13:08:16 GMT
content-type: application/json; charset=utf-8
content-length: 20
cf-ray: a354fb72292466e6-AMS
cf-cache-status: DYNAMIC
server: cloudflare

{"version":"0.33.2"}HTTP/2 200 
date: Thu, 03 Sep 2026 13:08:16 GMT
content-type: application/json; charset=utf-8
content-length: 20
cf-ray: a354fb755c08760b-AMS
cf-cache-status: DYNAMIC
access-control-allow-origin: *
server: cloudflare

{"version":"0.33.2"}

In [20]:
!tail -30 /tmp/ollama.log

srv  get_availabl: updating prompt cache
srv   prompt_save:  - saving prompt with length 294, total state size = 41.348 MiB (draft: 0.000 MiB)
srv          load:  - looking for better prompt, base f_keep = 0.054, f_sim = 0.055
srv          load:    - prompt with length     294, lcp =      16, f_keep = 0.054, f_sim = 0.055
srv        update:  - cache state: 1 prompts, 41.348 MiB (limits: 8192.000 MiB, 32768 tokens, 58248 est)
srv        update:    - prompt 0x1cd105f0:     294 tokens, checkpoints:  0,    41.348 MiB
srv  get_availabl: prompt cache update took 19.48 ms
slot launch_slot_: id  0 | task -1 | sampler chain: logits -> ?penalties -> ?dry -> ?top-n-sigma -> top-k -> ?typical -> top-p -> ?min-p -> ?xtc -> temp-ext -> dist 
slot launch_slot_: id  0 | task -1 | sampler params: 
	repeat_last_n = 64, repeat_penalty = 1.000, frequency_penalty = 0.000, presence_penalty = 0.000
	dry_multiplier = 0.000, dry_base = 1.750, dry_allowed_length = 2, dry_penalty_last_n = 64
	top_k = 20, top_p =